## BigQuery Setup

In [1]:
# ============================================================
# Cell 1: Authentication + BigQuery/GCS Clients
# ============================================================
import os
from google.oauth2 import service_account
from google.cloud import bigquery, storage

# ── Config ──────────────────────────────────────────────────
PROJECT_ID      = "recosys-489001"
DATASET_ID      = "recosys"
TABLE_ID        = "events_raw"
BUCKET_NAME     = "recosys-data-bucket"
RAW_PREFIX      = "raw/"
LOCATION        = "us-central1"          # BigQuery dataset location (free-tier friendly)
KEY_PATH        = os.path.expanduser("C:\\Users\\Patron\\Documents\\GitHub\\RecoSys\\secrets\\recosys-service-account.json")

# ── Credentials ─────────────────────────────────────────────
credentials = service_account.Credentials.from_service_account_file(
    KEY_PATH,
    scopes=[
        "https://www.googleapis.com/auth/cloud-platform",
        "https://www.googleapis.com/auth/bigquery",
    ],
)

bq_client  = bigquery.Client(project=PROJECT_ID, credentials=credentials)
gcs_client = storage.Client(project=PROJECT_ID, credentials=credentials)

print(f"✅ Authenticated as: {credentials.service_account_email}")
print(f"   BQ  project : {bq_client.project}")
print(f"   GCS project : {gcs_client.project}")

✅ Authenticated as: 921967012784-compute@developer.gserviceaccount.com
   BQ  project : recosys-489001
   GCS project : recosys-489001


## Creating BigQuery Dataset:

In [2]:
# ============================================================
# Cell 2: Create BigQuery Dataset
# ============================================================
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
dataset_ref.description = "REES46 eCommerce clickstream — recommendation system project"

try:
    dataset = bq_client.create_dataset(dataset_ref, exists_ok=True)
    print(f"✅ Dataset ready: {dataset.full_dataset_id}")
    print(f"   Location : {dataset.location}")
except Exception as e:
    print(f"❌ Failed to create dataset: {e}")
    raise

✅ Dataset ready: recosys-489001:recosys
   Location : us-central1


In [3]:
# ============================================================
# Cell 3: Load training months (Oct 2019 – Feb 2020) → events_raw
#         Holdout: Mar 2020, Apr 2020 — reserved for MLOps evaluation
# ============================================================
import time

# ── Train / holdout split ────────────────────────────────────
# Identify holdout files by stem name (matches GCS filename without extension)
HOLDOUT_STEMS = {"2020-Mar", "2020-Apr"}

bucket    = gcs_client.bucket(BUCKET_NAME)
all_blobs = sorted(b.name for b in bucket.list_blobs(prefix=RAW_PREFIX) if b.name.endswith(".csv"))

train_uris   = []
holdout_uris = []
for blob_name in all_blobs:
    stem = os.path.splitext(os.path.basename(blob_name))[0]   # e.g. "2019-Oct"
    uri  = f"gs://{BUCKET_NAME}/{blob_name}"
    (holdout_uris if stem in HOLDOUT_STEMS else train_uris).append(uri)

print("── File split ───────────────────────────────────────")
print(f"  Training ({len(train_uris)} files) : {[os.path.basename(u) for u in train_uris]}")
print(f"  Holdout  ({len(holdout_uris)} files) : {[os.path.basename(u) for u in holdout_uris]}")

# ── Schema ──────────────────────────────────────────────────
schema = [
    bigquery.SchemaField("event_time",    "TIMESTAMP"),
    bigquery.SchemaField("event_type",    "STRING"),
    bigquery.SchemaField("product_id",    "INTEGER"),
    bigquery.SchemaField("category_id",   "INTEGER"),
    bigquery.SchemaField("category_code", "STRING"),   # nullable
    bigquery.SchemaField("brand",         "STRING"),   # nullable
    bigquery.SchemaField("price",         "FLOAT64"),
    bigquery.SchemaField("user_id",       "INTEGER"),
    bigquery.SchemaField("user_session",  "STRING"),
]

job_config = bigquery.LoadJobConfig(
    schema=schema,
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,  # idempotent
    null_marker="",
    allow_jagged_rows=False,
    ignore_unknown_values=False,
    max_bad_records=1000,
)

table_ref = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

print(f"\n⏳ Starting load job ...")
print(f"   Source : {len(train_uris)} training CSVs (Oct 2019 – Feb 2020)")
print(f"   Target : {table_ref}")

start    = time.time()
load_job = bq_client.load_table_from_uri(train_uris, table_ref, job_config=job_config)
load_job.result()

elapsed = time.time() - start
print(f"\n✅ Load complete in {elapsed/60:.1f} min")
if load_job.errors:
    print(f"⚠️  Errors: {load_job.errors}")

── File split ───────────────────────────────────────
  Training (5 files) : ['2019-Dec.csv', '2019-Nov.csv', '2019-Oct.csv', '2020-Feb.csv', '2020-Jan.csv']
  Holdout  (2 files) : ['2020-Apr.csv', '2020-Mar.csv']

⏳ Starting load job ...
   Source : 5 training CSVs (Oct 2019 – Feb 2020)
   Target : recosys-489001.recosys.events_raw

✅ Load complete in 0.6 min


In [4]:
# ============================================================
# Cell 4: Verify load — row count + schema
# ============================================================

table = bq_client.get_table(table_ref)

print("── Schema ───────────────────────────────────────────")
for field in table.schema:
    print(f"  {field.name:<18} {field.field_type:<12} NULLABLE={field.is_nullable}")

print(f"\n── Table Stats ──────────────────────────────────────")
print(f"  Rows              : {table.num_rows:,}")
print(f"  Size on disk      : {table.num_bytes / 1e9:.2f} GB")
print(f"  Created           : {table.created}")
print(f"  Modified          : {table.modified}")

# Quick sanity query — should be very fast (metadata scan)
sanity_sql = f"""
SELECT
  COUNT(*)                                    AS total_rows,
  COUNT(DISTINCT user_id)                     AS unique_users,
  COUNT(DISTINCT product_id)                  AS unique_products,
  COUNTIF(event_type = 'view')                AS views,
  COUNTIF(event_type = 'cart')                AS carts,
  COUNTIF(event_type = 'purchase')            AS purchases,
  MIN(event_time)                             AS earliest_event,
  MAX(event_time)                             AS latest_event
FROM `{table_ref}`
"""

print(f"\n⏳ Running sanity query (will scan full table ~{table.num_bytes/1e9:.0f} GB)...")
result = bq_client.query(sanity_sql).result()

for row in result:
    print(f"\n── Sanity Check ─────────────────────────────────────")
    print(f"  Total rows        : {row.total_rows:,}")
    print(f"  Unique users      : {row.unique_users:,}")
    print(f"  Unique products   : {row.unique_products:,}")
    print(f"  Views             : {row.views:,}  ({row.views/row.total_rows*100:.1f}%)")
    print(f"  Carts             : {row.carts:,}  ({row.carts/row.total_rows*100:.1f}%)")
    print(f"  Purchases         : {row.purchases:,}  ({row.purchases/row.total_rows*100:.1f}%)")
    print(f"  Date range        : {row.earliest_event} → {row.latest_event}")

── Schema ───────────────────────────────────────────
  event_time         TIMESTAMP    NULLABLE=True
  event_type         STRING       NULLABLE=True
  product_id         INTEGER      NULLABLE=True
  category_id        INTEGER      NULLABLE=True
  category_code      STRING       NULLABLE=True
  brand              STRING       NULLABLE=True
  price              FLOAT        NULLABLE=True
  user_id            INTEGER      NULLABLE=True
  user_session       STRING       NULLABLE=True

── Table Stats ──────────────────────────────────────
  Rows              : 288,779,227
  Size on disk      : 32.03 GB
  Created           : 2026-03-06 02:10:45.210000+00:00
  Modified          : 2026-03-06 02:10:45.210000+00:00

⏳ Running sanity query (will scan full table ~32 GB)...

── Sanity Check ─────────────────────────────────────
  Total rows        : 288,779,227
  Unique users      : 11,839,964
  Unique products   : 313,884
  Views             : 271,045,030  (93.9%)
  Carts             : 12,877,066